# 252 — Clustering Recon Visualization (3D MNE Brain)

For each clustering CSV (e.g. `outputs/clustering/kmeans/labels_kmeans_raw_*.csv`):
- join each row with **fsaverage MNI coords** (`outputs/250_recon/fsaverage/coords/ALL_PATIENTS_contacts_fsaverage*.csv`)
- for each cluster, render the **fsaverage pial surface** (translucent) and overlay each electrode as a 3D sphere colored by group:
    - `cluster_<id>/by_patient/<view>.png` — colored by patient
    - `cluster_<id>/by_condition/<view>.png` — colored by condition
- save the merged-with-coords CSV and `UNMATCHED_contacts.csv` for QC

Views saved per cluster: `lateral_L`, `lateral_R`, `dorsal`, `frontal`.

Outputs go to: `outputs/250_recon/clustering_recon/<csv_basename>/cluster_<id>/<by_*>/`

**Toggles** (Cell A): `KEEP_WM`, `ALGOS`, `ALPHA`, `SPHERE_SCALE`, `SURF`, `CORTEX`.

**Contact-name join rule**
- **EL / BERN** (`EL030_audio_WM_ERSP_A_L10_TN.npy`) → contact `AL10`
- **PAT / GVA** (`PAT_3066_audio_WM_ERSP_AG10_TN.npy`) → contact `AG10`

> Uses **MNE-Python** + **PyVista** for 3D rendering (off-screen). Requires `mne`, `pyvista`, `pyvistaqt`.

In [1]:
# === Cell A: setup, paths, config (nilearn + MNE 3D, off-screen) ===
import os, sys, re, json, glob

clustering_folder = '210_kmeans_clustering'
clustering_folder = '230_blob_clustering_runs'
# clustering_folder = '231_minus101_clustering_runs'

# Off-screen env BEFORE any pyvista/mne import
os.environ['PYVISTA_OFF_SCREEN']      = 'true'
os.environ['MNE_3D_OPTION_ANTIALIAS'] = 'true'
os.environ['MESA_GL_VERSION_OVERRIDE'] = '3.3'

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import plotting as nlp

# ---- repo paths ----
NOTEBOOK_DIR = Path(os.getcwd()).resolve()
FUNCTIONS_DIR = NOTEBOOK_DIR / 'functions'
if str(FUNCTIONS_DIR) not in sys.path:
    sys.path.insert(0, str(FUNCTIONS_DIR))
import lf_recon_shared_config as C

# ---- I/O roots ----
OUT_ROOT          = Path(C.OUTPUTS_ROOT)
FSAV_COORDS_DIR   = OUT_ROOT / 'fsaverage' / 'coords'
# CLUSTERING_ROOT   = NOTEBOOK_DIR / 'outputs' / 'clustering'
CLUSTERING_ROOT   = NOTEBOOK_DIR / 'outputs' / clustering_folder
CLUSTERING_RECON  = OUT_ROOT / 'clustering_recon'/clustering_folder
CLUSTERING_RECON.mkdir(parents=True, exist_ok=True)

# ---- toggles ----
KEEP_WM      = False
ALGOS        = None
DPI          = 350

# ---- MNE 3D Brain config ----
SUBJECTS_DIR = Path('//nasac-m2.unige.ch/m-HumanNeuronLab/DATARAW/SEEG_EXPERIMENTS_BERN/Reconstruction')
SUBJECT      = 'fsaverage'
SURF         = 'pial'
CORTEX       = 'bone'
ALPHA        = 0.2
BACKGROUND   = 'white'
BRAIN_SIZE   = (1200, 1000)   # window size for off-screen render
SPHERE_SCALE = 0.4            # mm (smaller = smaller dots; tweak here)

VIEWS = {
    'lateral_L': dict(view='lateral', hemi='lh'),
    'lateral_R': dict(view='lateral', hemi='rh'),
    'dorsal'   : dict(view='dorsal',  hemi='both'),
    'frontal'  : dict(view='frontal', hemi='both'),
}

# ---- pyvista off-screen + MNE pyvista backend (no Qt) ----
import pyvista as pv
pv.OFF_SCREEN = True
# Set the GLOBAL theme window_size before creating the Brain — this is the
# size the off-screen render window will be created at on this pyvista version.
pv.global_theme.window_size            = list(BRAIN_SIZE)
pv.global_theme.background             = BACKGROUND
pv.global_theme.transparent_background = True
pv.global_theme.full_screen            = False
pv.global_theme.anti_aliasing          = 'msaa'

import mne
try:
    mne.viz.set_3d_backend('pyvista')
except Exception:
    mne.viz.set_3d_backend('notebook')
if hasattr(mne.viz, 'set_3d_options'):
    try:
        mne.viz.set_3d_options(antialias=True, depth_peeling=True, smooth_shading=True)
    except Exception:
        pass
os.environ['SUBJECTS_DIR'] = str(SUBJECTS_DIR)
print('mne     :', mne.__version__)
print('pyvista :', pv.__version__)
print('3D backend :', mne.viz.get_3d_backend())
print('global_theme.window_size :', pv.global_theme.window_size)

# sanity-check fsaverage
fsav_subj = SUBJECTS_DIR / SUBJECT
fsav_surf = fsav_subj / 'surf'
assert fsav_surf.exists() and (fsav_surf/'lh.pial').exists() and (fsav_surf/'rh.pial').exists(), \
    f'fsaverage surface files missing in {fsav_surf!r}'
print('fsaverage surf OK')

# ---- load fsaverage coords ----
fsav_csv = FSAV_COORDS_DIR / ('ALL_PATIENTS_contacts_fsaverage.csv' if KEEP_WM
                              else 'ALL_PATIENTS_contacts_fsaverage_nowm.csv')
assert fsav_csv.exists(), f'Missing {fsav_csv}'
fsav = pd.read_csv(fsav_csv)
print(f'\nLoaded fsaverage coords: {len(fsav)} contacts from {fsav_csv.name}')
fsav.head()

Using pyvistaqt 3d backend.

mne     : 1.6.1
pyvista : 0.43.4
3D backend : pyvistaqt
global_theme.window_size : [1200, 1000]
fsaverage surf OK


AssertionError: Missing \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\fsaverage\coords\ALL_PATIENTS_contacts_fsaverage_nowm.csv

In [ ]:
# === Cell B: discover clustering CSVs + define join rule ===

def _basename_no_ext(p):
    p = str(p).replace('\\\\', '/').replace('\\', '/')
    return os.path.basename(p).rsplit('.', 1)[0]

# BERN/EL  : <pat>_<cond>_WM_ERSP_<electrode>_<L|R><num>_TN... -> contact = electrode + side + num  (e.g. AL1, AR1)
# GVA/PAT  : <pat>_<cond>_WM_ERSP_<contact>_TN...          -> contact = electrode column directly
_RX_BERN = re.compile(r'_ERSP_([A-Za-z]+)_([LR])(\d+)_TN', re.IGNORECASE)
def contact_from_row(patient_id, electrode, file_path):
    base = _basename_no_ext(file_path)
    m = _RX_BERN.search(base)
    if m:
        return f'{m.group(1).upper()}{m.group(2).upper()}{m.group(3)}'
    if isinstance(electrode, str) and electrode.strip():
        return electrode.strip().upper()
    return None

if ALGOS is None:
    algo_dirs = sorted([p for p in CLUSTERING_ROOT.iterdir() if p.is_dir()]) if CLUSTERING_ROOT.exists() else []
else:
    algo_dirs = [CLUSTERING_ROOT / a for a in ALGOS]

clustering_csvs = []
for d in algo_dirs:
    if not d.exists():
        print(f'  [skip] {d} (missing)'); continue
    found = sorted(d.glob('labels_*.csv'))
    print(f'  {d.name}: {len(found)} labels CSV(s)')
    for f in found:
        clustering_csvs.append((d.name, f))

print(f'\nTotal clustering CSVs to process: {len(clustering_csvs)}')
for algo, f in clustering_csvs:
    print(f'  [{algo}]  {f.name}')

In [ ]:
# === Cell C: 3D translucent fsaverage brain + electrode spheres per cluster ===
from mne.viz import Brain

import matplotlib.colors as mcolors

def _safe_to_rgba(name, fallback=(0.5, 0.5, 0.5, 1.0)):
    """matplotlib.colors.to_rgba but with graceful fallback for unknown names (e.g. 'babyblue')."""
    try:
        return mcolors.to_rgba(name)
    except (ValueError, KeyError):
        return fallback

def _palette_patients(patient_ids):
    """Cohort-aware palette: EL patients use C.EL_COLOR_NAMES, PAT patients use C.PAT_COLOR_NAMES."""
    el_names  = list(getattr(C, 'EL_COLOR_NAMES',  []))
    pat_names = list(getattr(C, 'PAT_COLOR_NAMES', []))
    el_pats  = sorted({p for p in patient_ids if str(p).upper().startswith('EL')})
    pat_pats = sorted({p for p in patient_ids if str(p).upper().startswith('PAT')})
    other    = sorted({p for p in patient_ids if p not in el_pats and p not in pat_pats})
    palette  = {}
    for i, p in enumerate(el_pats):
        nm = el_names[i % len(el_names)] if el_names else 'tab:blue'
        palette[p] = _safe_to_rgba(nm)
    for i, p in enumerate(pat_pats):
        nm = pat_names[i % len(pat_names)] if pat_names else 'tab:red'
        palette[p] = _safe_to_rgba(nm)
    for i, p in enumerate(other):
        cmap = plt.get_cmap('tab10')
        palette[p] = cmap(i % 10)
    uniq = el_pats + pat_pats + other
    return palette, uniq

def _palette(values, cmap_name='tab10'):
    """Generic palette for non-patient grouping (e.g. condition)."""
    uniq = sorted(set(values))
    cmap = plt.get_cmap(cmap_name if len(uniq) <= 10 else 'tab20')
    n = max(len(uniq), 1)
    return {v: cmap(i / max(n-1, 1)) for i, v in enumerate(uniq)}, uniq

def _new_brain():
    try:
        return Brain(
            subject=SUBJECT, subjects_dir=str(SUBJECTS_DIR),
            surf=SURF, hemi='both',
            cortex=CORTEX, alpha=ALPHA,
            background=BACKGROUND, size=BRAIN_SIZE,
            offscreen=True, show=False, block=False,
        )
    except TypeError:
        return Brain(
            subject=SUBJECT, subjects_dir=str(SUBJECTS_DIR),
            surf=SURF, hemi='both',
            cortex=CORTEX, alpha=ALPHA,
            background=BACKGROUND, size=BRAIN_SIZE,
        )

def _force_size_and_render(brain):
    """Force the underlying pyvista plotter to honor BRAIN_SIZE and refresh GL."""
    try:
        plotter = brain._renderer.plotter
        plotter.window_size = list(BRAIN_SIZE)
        try:
            plotter.ren_win.SetSize(*BRAIN_SIZE)
        except Exception:
            pass
        plotter.reset_camera()
        plotter.render()
        return plotter
    except Exception as e:
        print(f'    [warn] could not resize plotter: {e}')
        return None

def _add_spheres(brain, coords_xyz_mm, colors_rgba, scale=SPHERE_SCALE):
    coords = np.asarray(coords_xyz_mm, dtype=float)
    for (xyz, c) in zip(coords, colors_rgba):
        hemi = 'lh' if xyz[0] < 0 else 'rh'
        try:
            brain.add_foci(
                xyz.reshape(1, 3),
                coords_as_verts=False,
                hemi=hemi,
                color=tuple(c[:3]),
                scale_factor=scale,
                alpha=1.0,
            )
        except Exception as e:
            print(f'    [warn] add_foci failed: {e}')

def _save_views(brain, out_dir, fname_prefix=''):
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = {}
    for tag, kw in VIEWS.items():
        try:
            if kw['hemi'] in ('lh', 'rh'):
                brain.show_view(view=kw['view'], hemi=kw['hemi'])
            else:
                brain.show_view(view=kw['view'])
        except TypeError:
            brain.show_view(view=kw['view'])
        plotter = _force_size_and_render(brain)
        out_png = out_dir / f'{fname_prefix}{tag}.png'
        try:
            # Use pyvista plotter.screenshot directly with explicit window_size — most reliable
            if plotter is not None:
                plotter.screenshot(filename=str(out_png),
                                   transparent_background=False,
                                   window_size=list(BRAIN_SIZE))
            else:
                brain.save_image(str(out_png))
        except Exception as e1:
            try:
                brain.save_image(str(out_png))
            except Exception as e2:
                print(f'    [warn] save failed for {tag}: {e1} / {e2}')
                continue
        paths[tag] = out_png
    return paths

def _save_legend(palette, uniq, title, out_png):
    fig, ax = plt.subplots(figsize=(4, max(2, 0.25*len(uniq) + 1)))
    ax.axis('off')
    handles = [plt.Line2D([0],[0], marker='o', linestyle='', color=palette[v],
                          label=str(v), markersize=10) for v in uniq]
    ax.legend(handles=handles, loc='center', frameon=False, fontsize=10,
              ncol=1 if len(uniq) <= 18 else 2, title=title)
    fig.tight_layout()
    fig.savefig(out_png, dpi=DPI, bbox_inches='tight')
    plt.close(fig)

summary_rows = []

for algo, csv_path in clustering_csvs:
    base = csv_path.stem
    out_dir = CLUSTERING_RECON / base
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f'\n=== [{algo}] {csv_path.name} -> {out_dir} ===')

    df = pd.read_csv(csv_path)
    cluster_cols = [c for c in df.columns if c.startswith('cluster_')]
    if not cluster_cols:
        print('  [skip] no cluster_* column'); continue
    cluster_col = cluster_cols[0]

    df['contact_name'] = df.apply(
        lambda r: contact_from_row(r['patient_id'], r['electrode'], r['file_path']),
        axis=1)
    # Case-insensitive join: uppercase the contact name on both sides
    df['contact_name'] = df['contact_name'].astype(str).str.upper()
    _fsav_join = fsav.copy()
    _fsav_join['name'] = _fsav_join['name'].astype(str).str.upper()
    merged = df.merge(
        _fsav_join[['patient', 'name', 'hemi', 'x', 'y', 'z', 'is_wm']],
        left_on=['patient_id', 'contact_name'],
        right_on=['patient', 'name'], how='left'
    )
    n_total   = len(merged)
    n_matched = int(merged['x'].notna().sum())
    n_missing = n_total - n_matched
    print(f'  rows: {n_total}  matched: {n_matched}  unmatched: {n_missing} ({100*n_missing/max(n_total,1):.1f}%)')

    if n_missing:
        unm = merged[merged['x'].isna()][['patient_id','condition','electrode','file_path','contact_name']].drop_duplicates()
        unm.to_csv(out_dir / 'UNMATCHED_contacts.csv', index=False)

    merged.to_csv(out_dir / f'{base}__with_fsaverage.csv', index=False)

    plot_df = merged.dropna(subset=['x','y','z']).copy()
    if not KEEP_WM and 'is_wm' in plot_df.columns:
        plot_df = plot_df[plot_df['is_wm'] == 0]
    if plot_df.empty:
        print('  [skip] no plottable rows'); continue

    clusters = sorted(plot_df[cluster_col].dropna().unique())
    print(f'  clusters to plot: {len(clusters)}')

    for cl in clusters:
        sub = plot_df[plot_df[cluster_col] == cl]
        cl_dir = out_dir / f'cluster_{cl}'
        coords_mm = sub[['x','y','z']].to_numpy(dtype=float)

        # ---- by patient ----
        pats = sub['patient_id'].astype(str).tolist()
        pal_p, uniq_p = _palette_patients(pats)
        cols_p = [pal_p[v] for v in pats]
        brain = _new_brain()
        _add_spheres(brain, coords_mm, cols_p)
        bp_dir = cl_dir / 'by_patient'
        _save_views(brain, bp_dir)
        _save_legend(pal_p, uniq_p, 'patient', bp_dir / 'legend.png')
        brain.close()

        # ---- by condition ----
        conds = sub['condition'].astype(str).tolist()
        pal_c, uniq_c = _palette(conds, cmap_name='tab10')
        cols_c = [pal_c[v] for v in conds]
        brain = _new_brain()
        _add_spheres(brain, coords_mm, cols_c)
        bc_dir = cl_dir / 'by_condition'
        _save_views(brain, bc_dir)
        _save_legend(pal_c, uniq_c, 'condition', bc_dir / 'legend.png')
        brain.close()

        summary_rows.append({
            'algo': algo, 'csv': base, 'cluster': cl,
            'n_contacts': len(sub),
            'n_patients': sub['patient_id'].nunique(),
            'n_conditions': sub['condition'].nunique(),
        })
        print(f'    cluster {cl}: {len(sub)} contacts -> {cl_dir}')

    print(f'  done -> {out_dir}')

if summary_rows:
    summ = pd.DataFrame(summary_rows)
    summ.to_csv(CLUSTERING_RECON / 'clustering_recon_summary.csv', index=False)
    print(f'\nSaved summary ({len(summ)} cluster rows)')
else:
    print('\nNo summary rows produced.')

In [ ]:
# === Debug: confirm L-vs-R hardcoding (and case sensitivity) cause unmatched contacts ===
# Re-runs the join using BOTH the old (L-only, case-sensitive) rule and the new (L|R, case-insensitive) rule
# and reports unmatched contacts per patient with samples + the FULL fsav contact list per patient.
import re as _re

_RX_OLD = _re.compile(r'_ERSP_([A-Za-z]+)_L(\d+)_TN', _re.IGNORECASE)
_RX_NEW = _re.compile(r'_ERSP_([A-Za-z]+)_([LR])(\d+)_TN', _re.IGNORECASE)

def _contact_old(electrode, file_path):
    base = _basename_no_ext(file_path)
    m = _RX_OLD.search(base)
    if m:
        return f'{m.group(1)}L{m.group(2)}'   # original: keeps original case
    if isinstance(electrode, str) and electrode.strip():
        return electrode.strip()
    return None

def _contact_new(electrode, file_path):
    base = _basename_no_ext(file_path)
    m = _RX_NEW.search(base)
    if m:
        return f'{m.group(1).upper()}{m.group(2).upper()}{m.group(3)}'   # uppercase normalized
    if isinstance(electrode, str) and electrode.strip():
        return electrode.strip().upper()
    return None

# Use the first clustering CSV for diagnosis
_algo, _csv = clustering_csvs[0]
df_dbg = pd.read_csv(_csv)
df_dbg['contact_old'] = df_dbg.apply(lambda r: _contact_old(r['electrode'], r['file_path']), axis=1)
df_dbg['contact_new'] = df_dbg.apply(lambda r: _contact_new(r['electrode'], r['file_path']), axis=1)

# OLD rule: case-sensitive join against fsav as-is
fsav_keys_old = set(zip(fsav['patient'].astype(str), fsav['name'].astype(str)))
# NEW rule: case-insensitive join (uppercase both sides)
fsav_keys_new = set(zip(fsav['patient'].astype(str), fsav['name'].astype(str).str.upper()))

df_dbg['matched_old'] = [(str(p), str(c)) in fsav_keys_old for p, c in zip(df_dbg['patient_id'], df_dbg['contact_old'])]
df_dbg['matched_new'] = [(str(p), str(c)) in fsav_keys_new for p, c in zip(df_dbg['patient_id'], df_dbg['contact_new'])]

print(f'CSV: {_csv.name}')
print(f'Total rows: {len(df_dbg)}')
print(f'  matched (OLD rule, L only, case-sensitive)  : {df_dbg["matched_old"].sum()}  unmatched: {(~df_dbg["matched_old"]).sum()}')
print(f'  matched (NEW rule, L|R, case-insensitive)   : {df_dbg["matched_new"].sum()}  unmatched: {(~df_dbg["matched_new"]).sum()}')

print('\nPer-patient unmatched count under OLD rule (top 20):')
print(df_dbg[~df_dbg['matched_old']].groupby('patient_id').size().sort_values(ascending=False).head(20))

recovered = df_dbg[(~df_dbg['matched_old']) & df_dbg['matched_new']]
print(f'\nRecovered by NEW rule: {len(recovered)} contacts')
print('Sample recovered (patient, contact_old -> contact_new, basename):')
for _, r in recovered.head(10).iterrows():
    print(f"  {r['patient_id']:>8}  {str(r['contact_old']):<10} -> {str(r['contact_new']):<10}  {_basename_no_ext(r['file_path'])}")

still = df_dbg[~df_dbg['matched_new']]
print(f'\nStill unmatched after NEW rule: {len(still)} contacts')
print('Sample still-unmatched (one row per patient + FULL fsav contact list for that patient):')
import pprint as _pp
for pid, grp in still.groupby('patient_id'):
    r = grp.iloc[0]
    fsav_for_p = sorted(fsav.loc[fsav['patient'].astype(str) == str(pid), 'name'].astype(str).str.upper().unique().tolist())
    print(f"  {pid:>8}  contact_new={r['contact_new']!r:<14}  basename={_basename_no_ext(r['file_path'])}")
    print(f"           fsav names ({len(fsav_for_p)}) for {pid}: {fsav_for_p}")